# Module 3 - Grounding with a Managed Knowledge Base

In this notebook you create an Amazon Bedrock Managed Knowledge Base from AnyCompany's travel policy, and use it so that both the travel chat and the expense checks answer from the actual policy, with citations.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [The travel policy documents](#documents)
4. [Upload the documents to Amazon S3](#upload)
5. [Create the Managed Knowledge Base](#create-kb)
6. [Connect the data source and start ingestion](#ingest)
7. [What the Managed Knowledge Base handles for you](#managed)
8. [Retrieve policy passages](#retrieve)
9. [Grounded chat answers with citations](#grounded-chat)
10. [Grounded expense checks](#grounded-expense)
11. [Try it](#try-it)
12. [Conclusion and key takeaways](#conclusion)
13. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

Your assistant has two gaps:

- **Travel chat** does not know AnyCompany's policy. In Module 1 it could not tell you the hotel limit in Chicago.
- **Expense checks** use rules you typed into the prompt by hand. If the policy changes, or the hand-typed rules are incomplete, the checks are wrong.

The fix for both is **retrieval-augmented generation (RAG)**: before the model answers, find the relevant passages in AnyCompany's policy documents and give them to the model along with the question. The model then answers from the policy instead of from memory, and can cite where each answer came from.

An [Amazon Bedrock Managed Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-build-managed.html) does the retrieval part for you. You point it at your documents, and it handles everything needed to search them:

| You do | The Managed Knowledge Base does |
|:--|:--|
| Put documents in a data source, here Amazon S3 | Parses and splits them into passages |
| Start ingestion | Converts passages into embeddings and stores them in a managed index |
| Send a question with the `Retrieve` API | Runs hybrid search, reranks the results, returns the best passages |
| Send the passages and question to a model with Converse | |

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients and set the names used in this module. The two model IDs are the ones you chose in Module 2.

In [ ]:
import glob
import json
import os
import re
import time

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

s3 = boto3.client("s3", region_name=REGION)
iam = boto3.client("iam")
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)  # Create and manage knowledge bases
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)  # Query knowledge bases
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)  # Call models

# Chosen in Module 2.
CHAT_MODEL_ID = "global.anthropic.claude-sonnet-5"
EXPENSE_MODEL_ID = "openai.gpt-oss-120b-1:0"

# Names for the resources you create. The account ID keeps the bucket name globally unique.
BUCKET_NAME = f"aim317-travel-policy-{ACCOUNT_ID}-{REGION}"
ROLE_NAME = "aim317-knowledge-base-role"
KNOWLEDGE_BASE_NAME = "aim317-travel-policy"

print(f"Region:         {REGION}")
print(f"Bucket:         {BUCKET_NAME}")
print(f"Knowledge base: {KNOWLEDGE_BASE_NAME}")

## 3. The travel policy documents <a id="documents"></a>

AnyCompany's travel policy is a set of short Markdown documents in `data/policies`. In a real company these might live in SharePoint, Confluence or Google Drive, all of which a Managed Knowledge Base can connect to directly.

Run the cell below to list the documents and read the hotel policy.

In [ ]:
POLICY_FILES = sorted(glob.glob("../data/policies/*.md"))

for path in POLICY_FILES:
    print(os.path.basename(path))

print("\n" + open("../data/policies/03-hotels.md").read())

Notice that the hotel policy has a higher limit for some cities, which the hand-typed rules in Module 2 did not include. Keep that in mind for the expense checks later.

## 4. Upload the documents to Amazon S3 <a id="upload"></a>

The knowledge base reads documents from a data source. Here the data source is an Amazon S3 bucket. Run the cell below to create the bucket, if it does not exist yet, and upload the documents.

In [ ]:
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except s3.exceptions.ClientError:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(Bucket=BUCKET_NAME, CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"Created bucket {BUCKET_NAME}")

for path in POLICY_FILES:
    s3.upload_file(path, BUCKET_NAME, os.path.basename(path))
print(f"Uploaded {len(POLICY_FILES)} documents")

The knowledge base also needs permission to read the bucket. It uses an [IAM service role](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-permissions.html) that Amazon Bedrock can assume, with read access to this bucket only.

Run the cell below to find the role, or create it if it does not exist.

In [ ]:
def get_or_create_knowledge_base_role():
    """Return the ARN of the knowledge base service role, creating the role if needed."""
    try:
        return iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        pass

    # Who can assume the role: Amazon Bedrock, for knowledge bases in this account only.
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/*"},
            },
        }],
    }
    # What the role can do: read the policy bucket.
    permissions = {
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": "s3:ListBucket", "Resource": f"arn:aws:s3:::{BUCKET_NAME}"},
            {"Effect": "Allow", "Action": "s3:GetObject", "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*"},
        ],
    }
    role = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="read-travel-policy-bucket",
                        PolicyDocument=json.dumps(permissions))
    time.sleep(10)  # Give IAM a moment to make the new role available.
    return role["Role"]["Arn"]


ROLE_ARN = get_or_create_knowledge_base_role()
print("Service role:", ROLE_ARN)

## 5. Create the Managed Knowledge Base <a id="create-kb"></a>

Creating a Managed Knowledge Base takes one API call. The `knowledgeBaseConfiguration` sets the type to `MANAGED` and the embedding model to `MANAGED`, so Amazon Bedrock chooses and runs the embedding model for you. There is no vector database to set up.

Run the cell below to create the knowledge base, or reuse it if you already created it, and wait until it is `ACTIVE`. This usually takes less than a minute.

In [ ]:
def find_knowledge_base(name):
    """Return the ID of the knowledge base with this name, or None."""
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]:
        if kb["name"] == name:
            return kb["knowledgeBaseId"]
    return None


KNOWLEDGE_BASE_ID = find_knowledge_base(KNOWLEDGE_BASE_NAME)

if KNOWLEDGE_BASE_ID is None:
    response = bedrock_agent.create_knowledge_base(
        name=KNOWLEDGE_BASE_NAME,
        description="AnyCompany travel and expense policy",
        roleArn=ROLE_ARN,
        knowledgeBaseConfiguration={
            "type": "MANAGED",
            "managedKnowledgeBaseConfiguration": {"embeddingModelType": "MANAGED"},
        },
    )
    KNOWLEDGE_BASE_ID = response["knowledgeBase"]["knowledgeBaseId"]
    print("Creating knowledge base", KNOWLEDGE_BASE_ID)

while (status := bedrock_agent.get_knowledge_base(knowledgeBaseId=KNOWLEDGE_BASE_ID)["knowledgeBase"]["status"]) == "CREATING":
    time.sleep(5)
print(f"Knowledge base {KNOWLEDGE_BASE_ID} is {status}")

## 6. Connect the data source and start ingestion <a id="ingest"></a>

The knowledge base is empty until you connect a data source and ingest it. The data source uses the managed S3 connector; `connectorParameters` names the bucket to read. Other connectors, such as SharePoint or Confluence, use the same call with different parameters.

Run the cell below to create the data source, if it does not exist yet, and start an **ingestion job**. Ingestion reads every document, splits it into passages, creates embeddings and indexes them. The call returns immediately; ingestion continues in the background.

In [ ]:
data_sources = bedrock_agent.list_data_sources(knowledgeBaseId=KNOWLEDGE_BASE_ID)["dataSourceSummaries"]

if data_sources:
    DATA_SOURCE_ID = data_sources[0]["dataSourceId"]
else:
    response = bedrock_agent.create_data_source(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        name="travel-policy-s3",
        dataDeletionPolicy="DELETE",  # Remove indexed data when the data source is deleted.
        dataSourceConfiguration={
            "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
            "managedKnowledgeBaseConnectorConfiguration": {
                "connectorParameters": {
                    "type": "S3",
                    "version": "1",
                    "connectionConfiguration": {
                        "bucketName": BUCKET_NAME,
                        "bucketOwnerAccountId": ACCOUNT_ID,
                    },
                    "deletionProtectionConfiguration": {"enableDeletionProtection": False},
                }
            },
        },
    )
    DATA_SOURCE_ID = response["dataSource"]["dataSourceId"]
    while bedrock_agent.get_data_source(knowledgeBaseId=KNOWLEDGE_BASE_ID, dataSourceId=DATA_SOURCE_ID)["dataSource"]["status"] == "CREATING":
        time.sleep(5)

ingestion_job = bedrock_agent.start_ingestion_job(knowledgeBaseId=KNOWLEDGE_BASE_ID, dataSourceId=DATA_SOURCE_ID)
INGESTION_JOB_ID = ingestion_job["ingestionJob"]["ingestionJobId"]
print(f"Data source {DATA_SOURCE_ID}: ingestion job {INGESTION_JOB_ID} started")

## 7. What the Managed Knowledge Base handles for you <a id="managed"></a>

Ingestion takes about two to three minutes for these documents. While it runs, here is what the Managed Knowledge Base does for you:

| Step | What happens |
|:--|:--|
| **Parsing** | Text and tables are extracted from each document (PDF, Word, HTML, Markdown and more) |
| **Chunking** | Documents are split into passages small enough to search and to fit in a prompt |
| **Embedding** | Each passage becomes a vector that represents its meaning |
| **Indexing** | Vectors and text are stored in a managed index |
| **Hybrid search** | Each query is matched both by meaning and by keywords |
| **Reranking** | The best candidates are rescored for relevance before being returned |

Run the cell below to wait for ingestion to finish. It checks every 15 seconds.

In [ ]:
while True:
    job = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KNOWLEDGE_BASE_ID, dataSourceId=DATA_SOURCE_ID, ingestionJobId=INGESTION_JOB_ID
    )["ingestionJob"]
    if job["status"] not in ("STARTING", "IN_PROGRESS"):
        break
    print(f"{time.strftime('%H:%M:%S')}  {job['status']}")
    time.sleep(15)

print(f"Ingestion {job['status']}")
stats = job.get("statistics", {})
print(f"Documents indexed: {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}, "
      f"failed: {stats.get('numberOfDocumentsFailed', 0)}")

## 8. Retrieve policy passages <a id="retrieve"></a>

With ingestion complete, you can search the policy. The [`Retrieve`](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_Retrieve.html) API takes a question and returns the most relevant passages, each with its source document and a relevance score. For a Managed Knowledge Base, you set the number of results in `managedSearchConfiguration`.

Run the cell below to search for the question your assistant could not answer in Module 1.

In [ ]:
def retrieve(query, number_of_results=5):
    """Return the most relevant policy passages for a query."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"managedSearchConfiguration": {"numberOfResults": number_of_results}},
    )
    return [
        {
            "text": result["content"]["text"],
            "source": result["location"]["s3Location"]["uri"].split("/")[-1],
            "score": result.get("score", 0),
        }
        for result in response["retrievalResults"]
    ]


for passage in retrieve("What is AnyCompany's nightly hotel limit in Chicago?", number_of_results=3):
    print(f"{passage['score']:.2f}  {passage['source']}")
    print("      " + passage["text"][:150].replace("\n", " ") + "...\n")

The hotel policy comes first, with a much higher score than the others. These documents are short, so each one fits in a single passage; longer documents are split into many passages, and only the relevant ones are returned.

## 9. Grounded chat answers with citations <a id="grounded-chat"></a>

Now combine retrieval with the chat model. The `ask_travel_assistant` function below:

1. Retrieves the most relevant policy passages for the question.
2. Numbers them and puts them in the prompt, inside `<policy>` tags.
3. Tells the model to answer only from those passages, and to cite them by number.
4. Returns the answer and the list of sources, so you can show employees where each answer came from.

In [ ]:
def get_text(response):
    """Return the text blocks of a Converse response, joined together."""
    return "".join(block["text"] for block in response["output"]["message"]["content"] if "text" in block)


def format_passages(passages):
    """Number the passages so the model can cite them."""
    return "\n\n".join(f"[{i}] (source: {p['source']})\n{p['text']}" for i, p in enumerate(passages, start=1))


GROUNDED_CHAT_PROMPT = (
    "You are the travel assistant for AnyCompany, helping employees plan and manage business "
    "travel. Answer using only the AnyCompany policy passages inside the <policy> tags. Cite the "
    "passages you use by number, like [1]. If the passages do not answer the question, say so and "
    "suggest contacting the AnyCompany Travel Desk. Be friendly and concise: three sentences or "
    "fewer unless the employee asks for detail."
)


def ask_travel_assistant(question):
    """Answer a travel question from the policy, and return the answer with its sources."""
    passages = retrieve(question)
    response = bedrock_runtime.converse(
        modelId=CHAT_MODEL_ID,
        system=[{"text": GROUNDED_CHAT_PROMPT}],
        messages=[{
            "role": "user",
            "content": [{"text": f"<policy>\n{format_passages(passages)}\n</policy>\n\nQuestion: {question}"}],
        }],
        inferenceConfig={"maxTokens": 2000},
    )
    return get_text(response), passages


def show_answer(question):
    answer, passages = ask_travel_assistant(question)
    print(f"Q: {question}\n\n{answer}\n")
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
    for n in cited:
        if 1 <= n <= len(passages):
            print(f"  [{n}] {passages[n - 1]['source']}")


show_answer("What is AnyCompany's nightly hotel limit in Chicago?")

The assistant now answers with AnyCompany's actual limit and cites the hotel policy. Run the cell below to ask the overbooked-hotel question from Module 2.

In [ ]:
show_answer(
    "My hotel in Chicago is overbooked and has no room for me tonight. "
    "I have a client meeting at 9am tomorrow. What should I do right now?"
)

This time the answer follows AnyCompany's trip disruption policy: the steps to take, the Travel Desk number, and the 25% exception for that night.

Finally, ask something the policy does not cover. A grounded assistant should say it does not know, rather than guess.

In [ ]:
show_answer("Can I bring my dog on a business trip?")

## 10. Grounded expense checks <a id="grounded-expense"></a>

Now apply the same idea to the expense checks. Instead of rules typed into the prompt, each check retrieves the relevant policy passages for that report.

A report can mix hotels, meals and flights, and one search for all of them at once can miss one of the policies. So `check_expense` runs **one search per expense type** in the report, and combines the results.

Run the cell below to load the expense reports from Module 2 and define the grounded check.

In [ ]:
with open("../data/expense_reports.json") as f:
    EXPENSE_REPORTS = json.load(f)


def format_report(report):
    """Turn an expense report into the text the model reads."""
    lines = [f"Expense report {report['id']} - {report['employee']} - {report['trip']}"]
    for item in report["items"]:
        lines.append(f"- {item['type']}: {item['description']}: ${item['amount']:.2f}")
    return "\n".join(lines)


GROUNDED_EXPENSE_PROMPT = (
    "You check AnyCompany expense reports against the AnyCompany policy passages inside the "
    "<policy> tags. Use only those passages. Reply with exactly two lines: first line APPROVED "
    "or FLAGGED, second line a one-sentence reason that cites the passage number, like [1]."
)


def get_verdict(text):
    """Find the first APPROVED or FLAGGED in a reply."""
    match = re.search(r"\b(APPROVED|FLAGGED)\b", text)
    return match.group(1) if match else "UNKNOWN"


def check_expense(report):
    """Check one expense report against the retrieved policy, and return the verdict and reason."""
    passages = []
    for item_type in sorted({item["type"] for item in report["items"]}):
        for passage in retrieve(f"AnyCompany {item_type} policy limits for a trip to {report['trip']}", 2):
            if passage["source"] not in [p["source"] for p in passages]:
                passages.append(passage)
    response = bedrock_runtime.converse(
        modelId=EXPENSE_MODEL_ID,
        system=[{"text": GROUNDED_EXPENSE_PROMPT}],
        messages=[{
            "role": "user",
            "content": [{"text": f"<policy>\n{format_passages(passages)}\n</policy>\n\n{format_report(report)}"}],
        }],
        inferenceConfig={"maxTokens": 2000},
    )
    text = get_text(response).strip()
    return get_verdict(text), text.splitlines()[-1]

Each report in `data/expense_reports.json` has two expected verdicts: `expected`, from the hand-typed rules in Module 2, and `expected_with_policy`, from the full policy. Run the cell below to check every report against the policy and compare.

In [ ]:
correct = 0
for report in EXPENSE_REPORTS:
    verdict, reason = check_expense(report)
    correct += verdict == report["expected_with_policy"]
    changed = "  <- changed from Module 2" if report["expected"] != report["expected_with_policy"] else ""
    print(f"{report['id']}  {verdict:<9} expected {report['expected_with_policy']:<9}{changed}")
    print(f"         {reason}\n")

print(f"{correct}/{len(EXPENSE_REPORTS)} match the policy")

Look at **EXP-002**, Marcus's three nights in New York at $320 a night. The hand-typed rule in Module 2 ($250 per night everywhere) flagged it. The actual policy allows $325 a night in New York, so it is approved, and the reason cites the hotel policy.

## 11. Try it <a id="try-it"></a>

Two things to try: let the assistant decide what to search, then change a policy and see the assistant follow.

### Let the assistant decide what to search

So far your code decided what to search: one search for each chat question, and one for each expense type. In an **agentic** setup, you give the model the search as a *tool*. The model decides whether to search, what to search for and how many times.

The cell below defines a `search_policy` tool and a short loop around the Converse API. When the model asks for a search, the loop runs `retrieve`, sends the passages back and lets the model continue. It also records each query, so you can see what the model chose to look up.

In [ ]:
# The tool the model can call. Its description tells the model when and how to use it.
SEARCH_TOOL = {
    "toolSpec": {
        "name": "search_policy",
        "description": (
            "Search the AnyCompany travel and expense policy. Use one focused query per topic, for "
            "example flights, hotels, meals, ground transportation, booking, expense reports or trip "
            "disruptions. Returns policy passages with their source document."
        ),
        "inputSchema": {"json": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "What to look up, for example 'hotel limit in London'"}},
            "required": ["query"],
        }},
    }
}

AGENT_PROMPT = (
    "You are the travel assistant for AnyCompany. Use the search_policy tool to look up the policy "
    "before you answer any question about it, with a separate search for each topic. Answer only "
    "from what the searches return, and name the policy documents you used, like (03-hotels.md). If "
    "the policy does not answer the question, say so and suggest contacting the AnyCompany Travel "
    "Desk. Be friendly and concise, and write plain text with no markdown or emojis."
)


def ask_agent(question, max_turns=6):
    """Let the model search the policy as often as it needs, and return the answer and its searches."""
    messages = [{"role": "user", "content": [{"text": question}]}]
    searches = []
    for _ in range(max_turns):
        response = bedrock_runtime.converse(
            modelId=CHAT_MODEL_ID,
            system=[{"text": AGENT_PROMPT}],
            messages=messages,
            toolConfig={"tools": [SEARCH_TOOL]},
            inferenceConfig={"maxTokens": 2000},
        )
        message = response["output"]["message"]
        messages.append(message)
        if response["stopReason"] != "tool_use":
            return get_text(response), searches
        # The model asked for one or more searches: run them and send the passages back.
        results = []
        for block in message["content"]:
            if "toolUse" in block:
                query = block["toolUse"]["input"]["query"]
                searches.append(query)
                passages = "\n\n".join(f"(source: {p['source']})\n{p['text']}" for p in retrieve(query, 3))
                results.append({"toolResult": {"toolUseId": block["toolUse"]["toolUseId"], "content": [{"text": passages}]}})
        messages.append({"role": "user", "content": results})
    return "I could not finish within the turn limit.", searches

In [ ]:
def show_agent_answer(question):
    answer, searches = ask_agent(question)
    print(f"Q: {question}\n")
    print(f"The model ran {len(searches)} search(es):")
    for query in searches:
        print(f"  - {query}")
    print(f"\n{answer}\n")


show_agent_answer("What is the daily meal limit for domestic travel?")
show_agent_answer(
    "I'm taking a 7-hour flight from New York to London and staying 3 nights. "
    "Can I fly business class, and what are my nightly hotel limit and daily meal limit?"
)

Look at the searches the model ran. The meal-limit question should need one search. The London question touches three policies, so expect separate searches for flights, hotels and meals, using queries the model wrote itself. Nothing in your code lists those topics. The exact queries and the number of searches can differ from run to run. That is the difference from the expense check in section 10 (Grounded expense checks), where your code looped over the expense types.

Try your own questions too, such as "Thanks, that's all I needed" (does the model search?) or "My flight home from Denver is cancelled. What do I do?". The [stretch notebook](stretch_agentcore.ipynb) builds the same idea as a Strands agent with a second tool, and deploys it.

### Change a policy

Policies change. Update one and see the assistant follow.

1. Open `data/policies/03-hotels.md`. Add Chicago to the high-cost cities in the table, at $325 a night, and remove Chicago from the sentence that lists the cities on the standard limit. Save the file.
2. Run the upload cell in section 4 (Upload the documents to Amazon S3), then the ingestion cells in section 6 (Connect the data source and start ingestion) and section 7 (What the Managed Knowledge Base handles for you).
3. Run the cell below. The answer should now reflect the new limit, without any change to your code or prompts.

In [ ]:
show_answer("What is AnyCompany's nightly hotel limit in Chicago?")

> **Note:** If you make this change, undo it before Module 4, so your results match the rest of the workshop. Run `!git checkout -- ../data/policies/03-hotels.md` in a new cell, then run the upload and ingestion cells again.

## 12. Conclusion and key takeaways <a id="conclusion"></a>

- **RAG** grounds a model's answers in your own documents: retrieve the relevant passages, then generate from them.
- A **Managed Knowledge Base** handles parsing, chunking, embeddings, indexing, hybrid search and reranking. You create it with one call, connect a data source, and ingest.
- **`Retrieve` plus Converse** gives you grounded answers from any model, with citations you can show to users. You can also give the model the search as a tool and let it decide what to look up.
- When the policy changes, you **re-ingest the documents**; the code and prompts stay the same.

## 13. Next steps <a id="next-steps"></a>

Your assistant now answers from AnyCompany's policy. In [Module 4 - Guardrails](04_guardrails.ipynb), you keep it safe and on-policy: redacting personal data such as passport numbers, refusing to help book outside policy, and checking that answers are supported by the passages you retrieved.

The knowledge base you created stays in your account for the rest of the workshop. The [Cleanup](99_cleanup.ipynb) notebook deletes it at the end.